# Fase 4 — Las 10 operaciones con Dask

**Proyecto:** BigData-Proy_Ciberseguridad · CIC-IoT-2023  
**Fase:** 4 — Procesamiento distribuido  
**Motor:** Dask

**Dask** representa el dataset como una colección de particiones y devuelve objetos *perezosos*: nada se calcula hasta que se pide un resultado. Las particiones se procesan en paralelo con hilos, y las operaciones que necesitan reagrupar filas (duplicados, `groupby`) provocan un intercambio tipo *shuffle*.

| Dato | Valor |
|---|---|
| Dataset | `01_fase1_datos/muestra/CICIoT2023_sample_600k.csv` |
| Registros | 600,000 |
| Columnas | 40 (39 numéricas + `Label`) |
| Clases | 34 (1 benigna + 33 de ataque) |
| Modo de ejecución | local, en la computadora del proyecto |
| Salida | `results/fase4/Dask/` |

Este notebook ejecuta las 10 operaciones del enunciado con **Dask** y guarda
cada resultado como CSV para poder compararlo con los demás motores.

## Las 10 operaciones del enunciado

| # | Operación | Requisito del enunciado | Archivo de salida |
|---|---|---|---|
| 01 | Carga y validación del dataset | Validación | `01_validacion.csv` |
| 02 | Limpieza de valores no válidos | Limpieza de datos | `02_limpieza.csv` |
| 03 | Tratamiento de duplicados | Eliminación de duplicados | `03_duplicados.csv` |
| 04 | Transformación de variables | Transformación de variables | `04_transformacion_variables.csv` |
| 05 | Filtrado de tráfico | Filtrado | `05_filtrado.csv` |
| 06 | Agregaciones globales | Agregaciones | `06_agregaciones.csv` |
| 07 | Agrupaciones por clase y protocolo | Agrupaciones | `07_agrupaciones.csv` |
| 08 | Ordenamiento y Top-10 | Ordenamiento | `08_ordenamiento_top10.csv` |
| 09 | Métricas de ciberseguridad | Cálculo de métricas | `09_metricas_ciberseguridad.csv` |
| 10 | Resumen consolidado por clase | CRUD y tablas de resultado | `10_resumen_consolidado.csv` |

## Reglas comunes a los cuatro motores

Para que la comparación sea justa, los cuatro notebooks aplican exactamente las
mismas reglas:

1. **Redondeo de ingesta.** Cada motor usa un parser de CSV distinto y los últimos
   decimales de un float pueden diferir en torno a `1e-11`. Todos redondean a
   **6 decimales** al leer el archivo.
2. **Valididad.** Una celda es *no válida* si está vacía, es `NaN` o es infinita.
   La operación 02 elimina las filas que contienen alguna celda no válida.
3. **Extremos.** `minimo_valido` y `maximo_valido` se calculan solo sobre valores
   finitos.
4. **Umbrales.** Los percentiles salen de `04_fase4_procesamiento/comun/umbrales.json`,
   calculado una vez con la biblioteca estándar, para que el filtro sea idéntico.
5. **Escritura.** Todos los CSV se escriben con el mismo formateador
   (`comun/io_comun.py`), así que se comparan celda por celda.

## Preparación del entorno

In [1]:
from __future__ import annotations

import sys
import time
from importlib.metadata import version
from pathlib import Path

AQUI = Path.cwd()
FASE = AQUI if (AQUI / "comun").exists() else AQUI.parent
if str(FASE) not in sys.path:
    sys.path.insert(0, str(FASE))

from comun import config
from comun.io_comun import escribir_csv, escribir_json, pct

import dask
import dask.dataframe as dd
import numpy as np
import pandas as pd

MOTOR = "dask"
VERSION = version("dask")
CSV_MUESTRA = config.CSV_MUESTRA
SALIDA = config.RUTA_RESULTADOS / MOTOR
SALIDA.mkdir(parents=True, exist_ok=True)
UMBRALES = config.cargar_umbrales()
TIEMPOS: list[dict] = []

# El archivo se parte en trozos de 16 MB y cada trozo se procesa en un hilo.
BLOQUE = "16mb"
HILOS = 4
dask.config.set(scheduler="threads", num_workers=HILOS)

# Tipos declarados de forma explicita: el motor no tiene que deducirlos y asi
# los cuatro motores parten del mismo esquema.
COLS_ENTERAS = [
    "Protocol Type", *config.COLS_CONTADORES, "Tot sum", "Min", "Max", "Number",
]
TIPOS = {
    columna: (
        "int64" if columna in COLS_ENTERAS
        else "object" if columna == "Label"
        else "float64"
    )
    for columna in config.COLUMNAS
}
FLOTANTES = [c for c, t in TIPOS.items() if t == "float64"]

print(f"Dask        {VERSION} (dask.dataframe)")
print(f"Python      {sys.version.split()[0]}")
print(f"Particiones {BLOQUE} por bloque, {HILOS} hilos, planificador threads")
print(f"Dataset     {CSV_MUESTRA.name} ({CSV_MUESTRA.stat().st_size:,} bytes)")
print(f"Resultados  {SALIDA}")


def medir(id_operacion: str, nombre: str, funcion):
    """Ejecuta la operación, cronometra y registra el tiempo."""
    inicio = time.perf_counter()
    resultado = funcion()
    segundos = round(time.perf_counter() - inicio, 3)
    TIEMPOS.append(
        {"motor": MOTOR, "id_operacion": id_operacion,
         "operacion": nombre, "segundos": segundos}
    )
    print(f"  -> [{id_operacion}] {nombre}: {segundos:.3f} s")
    return resultado


def csv(nombre: str, columnas: list[str], registros) -> None:
    """Escribe un resultado con el formateador común a los cuatro motores."""
    escribir_csv(SALIDA / nombre, columnas, registros)


def json_(nombre: str, contenido) -> None:
    escribir_json(SALIDA / nombre, contenido)


def _tipo(serie) -> str:
    """Tipo normalizado para que los cuatro motores coincidan.

    Dask convierte las columnas de texto a `string[pyarrow]`, asi que la
    pregunta correcta es si la columna es numerica, no si es de tipo `object`.
    """
    if not pd.api.types.is_numeric_dtype(serie):
        return "texto"
    if pd.api.types.is_integer_dtype(serie):
        return "entero"
    return "decimal"


def _texto(valor):
    """Vacio en el CSV cuando la columna no tiene ningun valor valido."""
    if valor is None or (isinstance(valor, float) and np.isnan(valor)):
        return ""
    return valor

Dask        2026.8.0 (dask.dataframe)
Python      3.12.3
Particiones 16mb por bloque, 4 hilos, planificador threads
Dataset     CICIoT2023_sample_600k.csv (124,498,206 bytes)
Resultados  C:\Users\steve\Desktop\Big_Data\ProyectoBigData_CICIoT2023\results\fase4\dask


## Carga del dataset

`read_csv` devuelve un DataFrame perezoso dividido en particiones de 16 MB. El redondeo a 6 decimales se encadena como una transformación más, así que también es perezoso.

**Cómo leer el tiempo de la carga.** Dask no lee el archivo en este paso: solo arma el grafo de tareas. Por eso el paso 00 aparece con milisegundos y el costo real se paga en la primera operación que fuerza la ejecución. Los tiempos de este notebook son comparables con los de los otros motores a partir del paso 01.

In [2]:
def cargar() -> dd.DataFrame:
    """Lee el CSV en particiones y normaliza la precisión."""
    datos = dd.read_csv(CSV_MUESTRA, blocksize=BLOQUE, dtype=TIPOS)
    return datos.assign(
        **{c: datos[c].round(config.REDONDEO_INGESTA) for c in FLOTANTES}
    )


df = medir("00", "Carga del dataset", cargar)

particiones = df.npartitions
print(f"Particiones: {particiones}    Columnas: {len(df.columns)}")
print(f"Filas: {len(df):,}    Primeras filas:")
df.head(5)

  -> [00] Carga del dataset: 0.060 s
Particiones: 7    Columnas: 40


Filas: 600,000    Primeras filas:


,Header_Length,Protocol Type,Time_To_Live,Rate,fin_flag_number,syn_flag_number,rst_flag_number,psh_flag_number,ack_flag_number,ece_flag_number,...,Tot sum,Min,Max,AVG,Std,Tot size,IAT,Number,Variance,Label
0,0.00,47,64.00,2109.927612,0.0,0.0,0.0,0.0,0.0,0.0,...,57800,578,578,578.00,0.000000,578.00,0.000475,100,0.000000,Mirai-greip_flood
1,0.00,47,64.00,5564.803906,0.0,0.0,0.0,0.0,0.0,0.0,...,57800,578,578,578.00,0.000000,578.00,0.000181,100,0.000000,Mirai-greip_flood
2,0.16,47,65.91,1789.707156,0.0,0.0,0.0,0.0,0.0,0.0,...,56464,60,578,564.64,77.409383,564.64,0.000560,100,5992.212525,Mirai-greip_flood
3,0.00,47,64.00,2923.022886,0.0,0.0,0.0,0.0,0.0,0.0,...,57800,578,578,578.00,0.000000,578.00,0.000400,100,0.000000,Mirai-greip_flood
4,0.32,47,63.36,600.243572,0.0,0.0,0.0,0.0,0.0,0.0,...,54588,60,578,545.88,119.524502,545.88,0.001667,100,14286.106667,Mirai-greip_flood


## Operación 01 — Carga y validación del dataset

**Requisito del enunciado:** Validación

Las estadísticas se calculan por partición con `map_partitions` y se combinan después. Los valores distintos se resuelven con `dask.delayed`, que recorre las particiones una sola vez y devuelve solo el resultado.

In [3]:
def _estadisticas(pdf: pd.DataFrame) -> pd.DataFrame:
    """Se ejecuta en cada partición y devuelve una fila por columna."""
    filas = []
    for columna in config.COLUMNAS:
        serie = pdf[columna]
        tipo = _tipo(serie)
        if tipo == "texto":
            filas.append(
                {"columna": columna, "celdas_no_validas": int(serie.isna().sum()),
                 "minimo_valido": float("nan"), "maximo_valido": float("nan")}
            )
            continue
        valores = serie.to_numpy()
        finitos = valores[np.isfinite(valores)]
        # `isna` ya cuenta los NaN, asi que aqui solo se suman los infinitos.
        invalidas = int(serie.isna().sum()) + int(np.isinf(valores).sum())
        minimo = maximo = float("nan")
        if finitos.size:
            minimo, maximo = float(finitos.min()), float(finitos.max())
        filas.append(
            {"columna": columna, "celdas_no_validas": invalidas,
             "minimo_valido": minimo, "maximo_valido": maximo}
        )
    return pd.DataFrame(filas)


META_01 = pd.DataFrame(
    {
        "columna": pd.Series([], dtype=object),
        "celdas_no_validas": pd.Series([], dtype="int64"),
        "minimo_valido": pd.Series([], dtype="float64"),
        "maximo_valido": pd.Series([], dtype="float64"),
    }
)


def op01():
    # Una particion por bloque: celdas no validas al alza y minimo y maximo
    # hacia extremos opuestos, nunca con la misma agregacion.
    por_particion = df.map_partitions(_estadisticas, meta=META_01)
    resumen = (
        por_particion.groupby("columna")
        .agg(
            celdas_no_validas=("celdas_no_validas", "sum"),
            minimo_valido=("minimo_valido", "min"),
            maximo_valido=("maximo_valido", "max"),
        )
        .compute()
    )

    # Los distintos se piden en un unico compute: Dask une los conjuntos de
    # cada particion y `nunique` es exacto. `dropna=False` para que el nulo
    # cuente como un valor mas, igual que hace `n_unique` de Polars.
    distintos = dict(
        zip(
            config.COLS_DISTINTOS,
            dask.compute(*[df[c].nunique(dropna=False) for c in config.COLS_DISTINTOS]),
        )
    )

    total = len(df)
    registros = []
    for columna in config.COLUMNAS:
        tipo = _tipo(df._meta[columna])
        celdas = int(resumen.loc[columna, "celdas_no_validas"])
        if tipo == "texto":
            minimo = maximo = ""
        elif tipo == "entero":
            minimo = int(resumen.loc[columna, "minimo_valido"])
            maximo = int(resumen.loc[columna, "maximo_valido"])
        else:
            minimo = _texto(resumen.loc[columna, "minimo_valido"])
            maximo = _texto(resumen.loc[columna, "maximo_valido"])
        registros.append(
            {
                "columna": columna,
                "tipo": tipo,
                "celdas_no_validas": celdas,
                "pct_no_validas": round(pct(celdas, total), 6),
                "distintos": int(distintos[columna]) if columna in distintos else "",
                "minimo_valido": minimo,
                "maximo_valido": maximo,
            }
        )
    csv("01_validacion.csv",
        ["columna", "tipo", "celdas_no_validas", "pct_no_validas", "distintos",
         "minimo_valido", "maximo_valido"],
        registros)
    json_("01_validacion_resumen.json",
          {"motor": MOTOR, "version": VERSION, "particiones": particiones,
           "bloque": BLOQUE, "hilos": HILOS, "filas": total,
           "columnas": len(config.COLUMNAS),
           "redondeo_decimales_ingesta": config.REDONDEO_INGESTA})
    return registros


validacion = medir("01", "Carga y validación del dataset", op01)
pd.DataFrame(validacion).head(12)


  -> [01] Carga y validación del dataset: 4.534 s


,columna,tipo,celdas_no_validas,pct_no_validas,distintos,minimo_valido,maximo_valido
0,Header_Length,decimal,0,0.000000,,0.0,60.0
1,Protocol Type,entero,0,0.000000,5,0,47
2,Time_To_Live,decimal,0,0.000000,,0.0,255.0
3,Rate,decimal,13,0.002167,99172,0.000428,7340032.0
4,fin_flag_number,decimal,0,0.000000,110,0.0,1.0
5,syn_flag_number,decimal,0,0.000000,145,0.0,1.0
6,rst_flag_number,decimal,0,0.000000,120,0.0,1.0
7,psh_flag_number,decimal,0,0.000000,143,0.0,1.0
8,ack_flag_number,decimal,0,0.000000,188,0.0,1.0
9,ece_flag_number,decimal,0,0.000000,11,0.0,0.6


## Operación 02 — Limpieza de valores no válidos

**Requisito del enunciado:** Limpieza de datos

`map_partitions` convierte cada valor no finito en `NaN` y descarta las filas incompletas. La limpieza sigue siendo una transformación perezosa: no se materializa hasta que una operación posterior pide un conteo.

In [4]:
@dask.delayed
def _invalidas_de_particion(pdf: pd.DataFrame) -> dict:
    """Celdas no validas de una partición, en una sola pasada."""
    salida = {}
    for columna in config.COLUMNAS:
        serie = pdf[columna]
        if TIPOS[columna] == "float64":
            # `isna` ya cuenta los NaN: aqui solo se suman los infinitos.
            valores = serie.to_numpy()
            salida[columna] = int(serie.isna().sum()) + int(np.isinf(valores).sum())
        else:
            salida[columna] = int(serie.isna().sum())
    return salida


@dask.delayed
def _nulos_por_particion(pdf: pd.DataFrame) -> dict:
    return {c: int(pdf[c].isna().sum()) for c in config.COLUMNAS}


def _limpiar_particion(pdf: pd.DataFrame) -> pd.DataFrame:
    """Todo valor no finito pasa a NaN y se eliminan las filas con faltantes."""
    copia = pdf.copy()
    for columna in FLOTANTES:
        valores = copia[columna].to_numpy()
        copia[columna] = np.where(np.isfinite(valores), valores, np.nan)
    return copia.dropna()


def op02():
    antes = len(df)
    parciales = dask.compute(*[_invalidas_de_particion(p) for p in df.to_delayed()])
    no_validas_antes = {c: sum(p[c] for p in parciales) for c in config.COLUMNAS}

    global limpio
    limpio = df.map_partitions(_limpiar_particion, meta=df._meta)
    despues = len(limpio)

    parciales = dask.compute(*[_nulos_por_particion(p) for p in limpio.to_delayed()])
    no_validas_despues = {c: sum(p[c] for p in parciales) for c in config.COLUMNAS}

    registros = [
        {"columna": c,
         "valores_no_validos_antes": no_validas_antes[c],
         "valores_no_validos_despues": no_validas_despues[c],
         "filas_eliminadas": antes - despues}
        for c in config.COLUMNAS if no_validas_antes[c] > 0
    ]
    csv("02_limpieza.csv",
        ["columna", "valores_no_validos_antes", "valores_no_validos_despues",
         "filas_eliminadas"],
        registros)
    json_("02_limpieza_resumen.json",
          {"motor": MOTOR,
           "criterio": "se elimina la fila con una celda vacía, NaN o infinita",
           "filas_antes": antes, "filas_despues": despues,
           "filas_eliminadas": antes - despues,
           "columnas_afectadas": registros})
    return limpio


limpio = medir("02", "Limpieza de valores no válidos", op02)
print(f"Filas antes: {len(df):,}   Filas después: {len(limpio):,}   "
      f"Eliminadas: {len(df) - len(limpio):,}")


  -> [02] Limpieza de valores no válidos: 4.500 s


Filas antes: 600,000   Filas después: 599,987   Eliminadas: 13


## Operación 03 — Tratamiento de duplicados

**Requisito del enunciado:** Eliminación de duplicados

`drop_duplicates()` provoca un *shuffle* para reunir las filas repetidas. Se miden las filas únicas por fila completa y por clave.

In [5]:
COLS_CLAVE = config.COLS_CLAVE_DUPLICADOS


def op03():
    antes = len(limpio)
    exactos = len(limpio.drop_duplicates())
    por_clave = len(limpio.drop_duplicates(subset=COLS_CLAVE))

    registros = [
        {"criterio": "filas_completas",
         "columnas_clave": f"las {len(config.COLUMNAS)} columnas",
         "filas_antes": antes, "filas_despues": exactos,
         "duplicados_eliminados": antes - exactos,
         "pct_duplicados": round(pct(antes - exactos, antes), 6)},
        {"criterio": "columnas_clave",
         "columnas_clave": ", ".join(COLS_CLAVE),
         "filas_antes": antes, "filas_despues": por_clave,
         "duplicados_eliminados": antes - por_clave,
         "pct_duplicados": round(pct(antes - por_clave, antes), 6)},
    ]
    csv("03_duplicados.csv",
        ["criterio", "columnas_clave", "filas_antes", "filas_despues",
         "duplicados_eliminados", "pct_duplicados"],
        registros)

    conteos = limpio.groupby(COLS_CLAVE).size().compute()
    conteos.name = "apariciones"
    ejemplos = conteos.reset_index().sort_values(
        ["apariciones", *COLS_CLAVE],
        ascending=[False, *([True] * len(COLS_CLAVE))],
    ).head(config.TOP_N)
    csv("03_duplicados_ejemplos.csv", ["apariciones"] + COLS_CLAVE,
        ejemplos.to_dict("records"))
    return registros


duplicados = medir("03", "Tratamiento de duplicados", op03)
pd.DataFrame(duplicados)

  -> [03] Tratamiento de duplicados: 7.508 s


,criterio,columnas_clave,filas_antes,filas_despues,duplicados_eliminados,pct_duplicados
0,filas_completas,las 40 columnas,599987,395496,204491,34.082572
1,columnas_clave,"Label, Protocol Type, Tot size, IAT, Rate, Number",599987,345902,254085,42.348418


## Operación 04 — Transformación de variables

**Requisito del enunciado:** Transformación de variables

El protocolo principal se calcula por partición con `map_partitions` y el resto con `assign`, de modo que las seis variables nuevas siguen siendo perezosas.

In [6]:
COLS_NUEVAS = ["size_kb", "rate_mbps", "coef_variacion", "total_flags",
              "protocolo_principal", "rango_iat"]

DEFINICIONES = {
    "size_kb": "Tot size / 1024 (kilobytes)",
    "rate_mbps": "Rate / 1 000 000 (paquetes por segundo)",
    "coef_variacion": "Std / AVG (dispersion del tamaño de paquete)",
    "total_flags": "Suma de los siete indicadores de flags TCP",
    "protocolo_principal": "Protocolo con valor 1 en las columnas one-hot",
    "rango_iat": "IAT clasificado con los percentiles 50 y 95: bajo, medio, alto",
}


def _protocolo_particion(pdf: pd.DataFrame) -> pd.DataFrame:
    """El protocolo principal es la PRIMERA columna one-hot con valor 1."""
    banderas = pdf[config.COLS_PROTOCOLO].to_numpy()
    nombres = np.array(config.COLS_PROTOCOLO, dtype=object)
    copia = pdf.copy()
    copia["protocolo_principal"] = np.where(
        banderas.max(axis=1) > 0, nombres[(banderas > 0).argmax(axis=1)], "otro"
    )
    return copia


def _rango_iat(d, p50: float, p95: float):
    return d["IAT"].map_partitions(
        lambda serie: pd.Series(
            np.where(serie <= p50, "bajo",
                     np.where(serie <= p95, "medio", "alto")),
            index=serie.index, name="rango_iat",
        ),
        meta=("rango_iat", "object"),
    )


def op04():
    iat = UMBRALES["IAT"]
    con_protocolo = limpio.map_partitions(
        _protocolo_particion,
        meta=limpio._meta.assign(protocolo_principal=pd.Series([], dtype=object)),
    )
    trabajo = con_protocolo.assign(
        size_kb=lambda d: d["Tot size"] / 1024,
        rate_mbps=lambda d: d["Rate"] / 1_000_000,
        coef_variacion=lambda d: (d["Std"] / d["AVG"]).where(d["AVG"] > 0),
        total_flags=lambda d: d[config.COLS_FLAGS].sum(axis=1),
        rango_iat=lambda d: _rango_iat(d, iat["p50"], iat["p95"]),
    )

    # Un solo `dask.compute` con todas las estadisticas: Dask fusiona los grafos
    # y recorre las particiones una unica vez en lugar de una vez por columna.
    textos = {columna: _tipo(trabajo[columna]) == "texto" for columna in COLS_NUEVAS}
    pedidos = []
    for columna in COLS_NUEVAS:
        serie = trabajo[columna]
        pedidos.append(serie.isna().sum())
        pedidos += (
            [serie.nunique()] if textos[columna]
            else [serie.min(), serie.max(), serie.mean()]
        )
    valores = dask.compute(*pedidos)

    registros = []
    posicion = 0
    for columna in COLS_NUEVAS:
        nulos = int(valores[posicion]); posicion += 1
        if textos[columna]:
            distintos = int(valores[posicion]); posicion += 1
            minimo = maximo = media = ""
        else:
            minimo, maximo, media = (float(v) for v in valores[posicion:posicion + 3])
            posicion += 3
            distintos = ""
        registros.append(
            {
                "columna": columna,
                "tipo": "texto" if textos[columna] else _tipo(trabajo[columna]),
                "nulos": nulos,
                "minimo": minimo,
                "maximo": maximo,
                "media": media,
                "distintos": distintos,
                "definicion": DEFINICIONES[columna],
            }
        )
    csv("04_transformacion_variables.csv",
        ["columna", "tipo", "nulos", "minimo", "maximo", "media", "distintos",
         "definicion"],
        registros)

    # En Dask `head()` ya materializa el resultado: devuelve un DataFrame de pandas.
    muestra = trabajo[
        ["Label", "Protocol Type", "Rate", "Tot size", "IAT", *COLS_NUEVAS]
    ].head(config.FILAS_MUESTRA_TRANSFORMACION)
    csv("04_transformacion_muestra.csv", list(muestra.columns),
        muestra.to_dict("records"))
    return trabajo, registros


transformado, variables = medir("04", "Transformación de variables", op04)
pd.DataFrame(variables)

  -> [04] Transformación de variables: 2.347 s


,columna,tipo,nulos,minimo,maximo,media,distintos,definicion
0,size_kb,decimal,0,0.044922,4.645605,0.128457,,Tot size / 1024 (kilobytes)
1,rate_mbps,decimal,0,0.0,7.340032,0.028514,,Rate / 1 000 000 (paquetes por segundo)
2,coef_variacion,decimal,0,0.0,5.916701,0.105852,,Std / AVG (dispersion del tamaño de paquete)
3,total_flags,decimal,0,0.0,2.38,0.613194,,Suma de los siete indicadores de flags TCP
4,protocolo_principal,texto,0,,,,13,Protocolo con valor 1 en las columnas one-hot
5,rango_iat,texto,0,,,,3,IAT clasificado con los percentiles 50 y 95: b...


In [7]:
# A partir de aqui df pasa a ser el dataset limpio y transformado:
# las operaciones 05 a 10 ya pueden usar las columnas nuevas.
df = transformado
print(f"Columnas del dataset transformado: {len(df.columns)}")

Columnas del dataset transformado: 46


## Operación 05 — Filtrado de tráfico

**Requisito del enunciado:** Filtrado

Las máscaras booleanas se construyen de forma perezosa. Los cuatro conteos se piden en un único `dask.compute`, así que el archivo se recorre una sola vez.

In [8]:
def op05():
    total = len(df)
    rate, size, iat = UMBRALES["Rate"], UMBRALES["Tot size"], UMBRALES["IAT"]

    filtros = [
        ("trafico_alto", f"Rate >= p95 ({rate['p95']:.6f})", rate["p95"],
         df["Rate"] >= rate["p95"]),
        ("paquetes_grandes", f"Tot size >= p95 ({size['p95']:.6f})", size["p95"],
         df["Tot size"] >= size["p95"]),
        ("trafico_intenso",
         f"Rate >= p95 ({rate['p95']:.6f}) y Tot size >= p50 ({size['p50']:.6f})",
         rate["p95"],
         (df["Rate"] >= rate["p95"]) & (df["Tot size"] >= size["p50"])),
        ("iat_reducido", f"0 < IAT <= p95 ({iat['p95']:.6f})", iat["p95"],
         (df["IAT"] > 0) & (df["IAT"] <= iat["p95"])),
    ]

    conteos = dask.compute(*[mascara.sum() for _, _, _, mascara in filtros])
    registros = []
    for (nombre, condicion, umbral, _), encontradas in zip(filtros, conteos):
        encontradas = int(encontradas)
        registros.append(
            {"filtro": nombre, "condicion": condicion, "umbral": umbral,
             "filas_encontradas": encontradas,
             "pct_del_total": round(pct(encontradas, total), 6)}
        )
    csv("05_filtrado.csv",
        ["filtro", "condicion", "umbral", "filas_encontradas", "pct_del_total"],
        registros)

    intenso = df[(df["Rate"] >= rate["p95"]) & (df["Tot size"] >= size["p50"])]
    conteos = intenso.groupby("Label").size().compute().reset_index(name="registros")
    agregados = (
        intenso.groupby("Label")
        .agg({"Tot size": "sum", "Rate": "mean"})
        .compute()
        .reset_index()
        .rename(columns={"Tot size": "volumen_bytes", "Rate": "media_rate"})
    )
    por_label = (
        conteos.merge(agregados, on="Label")
        .sort_values(["registros", "Label"], ascending=[False, True])
    )
    csv("05_filtrado_por_label.csv",
        ["Label", "registros", "volumen_bytes", "media_rate"],
        por_label.to_dict("records"))
    return registros


filtrado = medir("05", "Filtrado de tráfico", op05)
pd.DataFrame(filtrado)

  -> [05] Filtrado de tráfico: 5.661 s


,filtro,condicion,umbral,filas_encontradas,pct_del_total
0,trafico_alto,Rate >= p95 (63492.340297),63492.340297,30029,5.004942
1,paquetes_grandes,Tot size >= p95 (586.680000),586.680000,31235,5.205946
2,trafico_intenso,Rate >= p95 (63492.340297) y Tot size >= p50 (...,63492.340297,30013,5.002275
3,iat_reducido,0 < IAT <= p95 (0.001070),0.001070,569989,95.000225


## Operación 06 — Agregaciones globales

**Requisito del enunciado:** Agregaciones

**Limitaciones de Dask.** La mediana exacta no está implementada y `agg` no acepta una lista de funciones. Por eso las seis reducciones nativas se piden juntas en un único `dask.compute` y la mediana se arma con NumPy sobre los bloques ya calculados.

Los dos grupos viajan en `compute` separados: mezclarlos en una sola llamada de 310 tareas hace que Dask dedique más tiempo a optimizar el grafo que a calcular, y la operación pasa de 4 s a más de 70 s.

In [9]:
@dask.delayed
def _bloques(pdf: pd.DataFrame) -> dict:
    """Un bloque con los vectores de todas las columnas indicadoras."""
    return {columna: pdf[columna].to_numpy() for columna in config.COLS_INDICADORES}


ORDEN = ("conteo", "suma", "media", "mediana", "minimo", "maximo",
         "desviacion_estandar")


def op06():
    nativas = ("conteo", "suma", "media", "minimo", "maximo", "desviacion_estandar")

    # Las seis reducciones nativas se piden juntas en un unico compute, asi el
    # archivo se recorre una sola vez.
    reducciones = []
    for columna in config.COLS_INDICADORES:
        serie = df[columna]
        for metrica, expresion in (
            ("conteo", serie.count()),
            ("suma", serie.sum()),
            ("media", serie.mean()),
            ("minimo", serie.min()),
            ("maximo", serie.max()),
            ("desviacion_estandar", serie.std()),
        ):
            reducciones.append((columna, metrica, expresion))
    resultados = dask.compute(*[tarea for _, _, tarea in reducciones])

    # La mediana se arma con los bloques. Va en un segundo compute independiente:
    # mezclar las dos listas en un solo compute obliga a Dask a optimizar un
    # grafo muy grande y la operacion pasa de 4 s a mas de 70 s.
    bloques = dask.compute(*[_bloques(particion) for particion in df.to_delayed()])

    registros = []
    for indice, columna in enumerate(config.COLS_INDICADORES):
        grupo = dict(zip(
            nativas,
            resultados[indice * len(nativas): (indice + 1) * len(nativas)],
        ))
        partes = [bloque[columna] for bloque in bloques]
        grupo["mediana"] = float(np.median(np.concatenate(partes)))
        for metrica in ORDEN:
            registros.append(
                {"columna": columna, "metrica": metrica, "valor": grupo[metrica]}
            )

    csv("06_agregaciones.csv", ["columna", "metrica", "valor"], registros)
    return registros


agregaciones = medir("06", "Agregaciones globales", op06)
pd.DataFrame(agregaciones).head(14)

  -> [06] Agregaciones globales: 3.202 s


,columna,metrica,valor
0,Rate,conteo,5.999870e+05
1,Rate,suma,1.710832e+10
2,Rate,media,2.851448e+04
3,Rate,mediana,2.466222e+04
4,Rate,minimo,4.280000e-04
5,Rate,maximo,7.340032e+06
6,Rate,desviacion_estandar,3.262685e+04
7,Tot size,conteo,5.999870e+05
8,Tot size,suma,7.892257e+07
9,Tot size,media,1.315405e+02


## Operación 07 — Agrupaciones por clase y protocolo

**Requisito del enunciado:** Agrupaciones

`groupby` con las llaves `Label` y `Protocol Type`. Dask resuelve el regrouping con un *shuffle* y devuelve un DataFrame de pandas al pedir `.compute()`; como el resultado son pocas filas, el orden final se aplica en el driver.

In [10]:
def op07():
    total = len(df)
    conteos = df.groupby(["Label", "Protocol Type"]).size().compute()
    conteos = conteos.reset_index(name="registros")
    agregados = (
        df.groupby(["Label", "Protocol Type"])
        .agg({"Tot size": "sum", "Rate": "mean", "IAT": "mean"})
        .compute()
        .reset_index()
        .rename(columns={"Tot size": "volumen_bytes", "Rate": "media_rate",
                         "IAT": "media_iat"})
    )
    grupos = (
        conteos.merge(agregados, on=["Label", "Protocol Type"])
        .sort_values(["Label", "Protocol Type"])
    )
    registros = [
        {"Label": f["Label"], "Protocol Type": int(f["Protocol Type"]),
         "registros": int(f["registros"]),
         "pct_registros": round(pct(int(f["registros"]), total), 6),
         "volumen_bytes": f["volumen_bytes"],
         "media_tot_size": f["volumen_bytes"] / int(f["registros"]),
         "media_rate": f["media_rate"], "media_iat": f["media_iat"]}
        for _, f in grupos.iterrows()
    ]
    csv("07_agrupaciones.csv",
        ["Label", "Protocol Type", "registros", "pct_registros", "volumen_bytes",
         "media_tot_size", "media_rate", "media_iat"],
        registros)
    return registros


agrupaciones = medir("07", "Agrupaciones por clase y protocolo", op07)
pd.DataFrame(agrupaciones).head(10)

  -> [07] Agrupaciones por clase y protocolo: 4.197 s


,Label,Protocol Type,registros,pct_registros,volumen_bytes,media_tot_size,media_rate,media_iat
0,Backdoor_Malware,6,28,0.004667,1.013480e+04,361.957143,673.427166,0.026105
1,Backdoor_Malware,17,14,0.002333,2.075000e+03,148.214286,48.065148,0.038464
2,Benign,0,31,0.005167,4.701800e+03,151.670968,221.481694,0.019105
3,Benign,1,3,0.000500,4.088000e+02,136.266667,128.237094,0.018639
4,Benign,6,12885,2.147547,8.392353e+06,651.327377,2901.537006,0.008740
5,Benign,17,1166,0.194338,1.785024e+05,153.089537,152.587589,0.013828
6,BrowserHijacking,6,67,0.011167,3.970520e+04,592.614925,23350.552569,0.015639
7,BrowserHijacking,17,9,0.001500,2.711400e+03,301.266667,259.118631,0.014353
8,CommandInjection,6,54,0.009000,3.390350e+04,627.842593,2927.717130,0.015490
9,CommandInjection,17,16,0.002667,2.167700e+03,135.481250,63.663114,0.029455


## Operación 08 — Ordenamiento y Top-10

**Requisito del enunciado:** Ordenamiento

Resumen por clase ordenado por volumen descendente. El `groupby` es perezoso y el `sort_values` se aplica sobre las 34 filas ya traídas al driver.

In [11]:
def _resumen_por_clase() -> pd.DataFrame:
    """Volumen y media de tasa por clase, ordenado por volumen."""
    conteos = df.groupby("Label").size().compute().reset_index(name="registros")
    agregados = (
        df.groupby("Label")
        .agg({"Tot size": "sum", "Rate": "mean"})
        .compute()
        .reset_index()
        .rename(columns={"Tot size": "volumen_bytes", "Rate": "media_rate"})
    )
    return (
        conteos.merge(agregados, on="Label")
        .sort_values(["volumen_bytes", "Label"], ascending=[False, True])
    )


def op08():
    volumen_total = float(df["Tot size"].sum().compute())
    top = _resumen_por_clase().head(config.TOP_N)
    registros = [
        {"posicion": posicion, "Label": f["Label"],
         "registros": int(f["registros"]), "volumen_bytes": f["volumen_bytes"],
         "volumen_mb": f["volumen_bytes"] / (1024 * 1024),
         "pct_volumen": round(pct(float(f["volumen_bytes"]), volumen_total), 6),
         "media_rate": f["media_rate"]}
        for posicion, (_, f) in enumerate(top.iterrows(), start=1)
    ]
    csv("08_ordenamiento_top10.csv",
        ["posicion", "Label", "registros", "volumen_bytes", "volumen_mb",
         "pct_volumen", "media_rate"],
        registros)
    return registros


top10 = medir("08", "Ordenamiento y Top-10", op08)
pd.DataFrame(top10)

  -> [08] Ordenamiento y Top-10: 4.186 s


,posicion,Label,registros,volumen_bytes,volumen_mb,pct_volumen,media_rate
0,1,Benign,14085,8.575966e+06,8.178679,10.866303,2667.481157
1,2,Mirai-greeth_flood,12719,7.433445e+06,7.089085,9.418655,5576.009272
2,3,Mirai-udpplain,11423,6.243730e+06,5.954485,7.911209,6132.884517
3,4,DDoS-ICMP_Flood,92356,5.610000e+06,5.350113,7.108233,39957.267678
4,5,Mirai-greip_flood,9642,5.451962e+06,5.199396,6.907988,5055.603642
5,6,DDoS-ICMP_Fragmentation,5804,5.140472e+06,4.902337,6.513311,2973.919964
6,7,DDoS-UDP_Flood,69419,4.218632e+06,4.023201,5.345279,33345.355260
7,8,DDoS-TCP_Flood,57687,3.631696e+06,3.463455,4.601593,32271.151585
8,9,DoS-UDP_Flood,39414,3.516836e+06,3.353916,4.456058,20014.109743
9,10,DDoS-UDP_Fragmentation,3681,3.288975e+06,3.136611,4.167344,2026.716545


## Operación 09 — Métricas de ciberseguridad

**Requisito del enunciado:** Cálculo de métricas

Indicadores de población, volumen, comportamiento de flags, concentración y calidad del dataset.

In [12]:
def op09():
    benignos = int((df["Label"] == config.ETIQUETA_BENIGNA).sum().compute())
    total = len(df)
    ataques = total - benignos
    volumen_total = float(df["Tot size"].sum().compute())
    paquetes = float(df["Number"].sum().compute())
    duplicados = total - len(df.drop_duplicates())
    clases = int(df["Label"].nunique().compute())
    clases_ataque = int(
        df[df["Label"] != config.ETIQUETA_BENIGNA]["Label"].nunique().compute()
    )

    # Las medias de flags en una sola pasada.
    medias = dask.compute(
        df["syn_flag_number"].mean(), df["ack_flag_number"].mean(),
        df["rst_flag_number"].mean(), df["fin_flag_number"].mean(),
        df["IAT"].mean(), df["Rate"].mean(),
    )
    media_syn, media_ack, media_rst, media_fin, iat_medio, tasa_media = medias

    por_clase = _resumen_por_clase()
    principal = por_clase.iloc[0]
    top5 = por_clase.head(5)

    metricas = [
        ("poblacion", "total_registros", total),
        ("poblacion", "total_clases", clases),
        ("poblacion", "clases_de_ataque", clases_ataque),
        ("poblacion", "registros_benignos", benignos),
        ("poblacion", "registros_de_ataque", ataques),
        ("poblacion", "pct_registros_benignos", round(pct(benignos, total), 6)),
        ("poblacion", "pct_registros_ataque", round(pct(ataques, total), 6)),
        ("volumen", "volumen_total_bytes", volumen_total),
        ("volumen", "volumen_total_mb", volumen_total / (1024 * 1024)),
        ("volumen", "paquetes_totales", paquetes),
        ("volumen", "tamano_medio_paquete_bytes",
         volumen_total / paquetes if paquetes else 0.0),
        ("volumen", "tasa_media", float(tasa_media)),
        ("comportamiento", "media_syn_flag", float(media_syn)),
        ("comportamiento", "media_ack_flag", float(media_ack)),
        ("comportamiento", "media_rst_flag", float(media_rst)),
        ("comportamiento", "media_fin_flag", float(media_fin)),
        ("comportamiento", "ratio_syn_ack",
         float(media_syn) / float(media_ack) if media_ack else 0.0),
        ("comportamiento", "iat_medio", float(iat_medio)),
        ("concentracion", "clase_principal", principal["Label"]),
        ("concentracion", "pct_registros_clase_principal",
         round(pct(int(principal["registros"]), total), 6)),
        ("concentracion", "pct_registros_top5",
         round(pct(int(top5["registros"].sum()), total), 6)),
        ("concentracion", "pct_volumen_top5",
         round(pct(float(top5["volumen_bytes"].sum()), volumen_total), 6)),
        ("calidad", "filas_duplicadas", duplicados),
        ("calidad", "pct_filas_duplicadas", round(pct(duplicados, total), 6)),
        ("calidad", "pct_filas_unicas", round(pct(total - duplicados, total), 6)),
    ]
    registros = [{"categoria": c, "metrica": m, "valor": v} for c, m, v in metricas]
    csv("09_metricas_ciberseguridad.csv", ["categoria", "metrica", "valor"], registros)
    return registros


metricas = medir("09", "Métricas de ciberseguridad", op09)
pd.DataFrame(metricas)

  -> [09] Métricas de ciberseguridad: 15.916 s


,categoria,metrica,valor
0,poblacion,total_registros,599987
1,poblacion,total_clases,34
2,poblacion,clases_de_ataque,33
3,poblacion,registros_benignos,14085
4,poblacion,registros_de_ataque,585902
5,poblacion,pct_registros_benignos,2.347551
6,poblacion,pct_registros_ataque,97.652449
7,volumen,volumen_total_bytes,78922573.190802
8,volumen,volumen_total_mb,75.266431
9,volumen,paquetes_totales,57290468.0


## Operación 10 — Resumen consolidado por clase

**Requisito del enunciado:** CRUD y tablas de resultado

Tabla final por clase con el conteo, el peso porcentual, el volumen y las medias de las variables transformadas.

In [13]:
def op10():
    total = len(df)
    volumen_total = float(df["Tot size"].sum().compute())
    conteos = df.groupby("Label").size().compute().reset_index(name="registros")
    agregados = (
        df.groupby("Label")
        .agg({"Tot size": "sum", "Rate": "mean"})
        .compute()
        .reset_index()
        .rename(columns={"Tot size": "volumen_bytes", "Rate": "media_rate"})
    )
    medias = (
        df.groupby("Label")
        .agg({"Tot size": "mean", "IAT": "mean", "size_kb": "mean",
              "syn_flag_number": "mean", "ack_flag_number": "mean",
              "rst_flag_number": "mean", "total_flags": "mean"})
        .compute()
        .reset_index()
        .rename(columns={"Tot size": "media_tot_size", "IAT": "media_iat",
                         "size_kb": "media_size_kb", "syn_flag_number": "media_syn",
                         "ack_flag_number": "media_ack",
                         "rst_flag_number": "media_rst",
                         "total_flags": "media_total_flags"})
    )
    protocolos = (
        df.groupby("Label")["Protocol Type"].nunique().compute()
        .reset_index(name="protocolos_distintos")
    )
    resumen = (
        conteos.merge(agregados, on="Label")
        .merge(medias, on="Label")
        .merge(protocolos, on="Label")
        .sort_values(["registros", "Label"], ascending=[False, True])
    )
    registros = [
        {"posicion": posicion, "Label": f["Label"], "registros": int(f["registros"]),
         "pct_registros": round(pct(int(f["registros"]), total), 6),
         "volumen_bytes": f["volumen_bytes"],
         "pct_volumen": round(pct(float(f["volumen_bytes"]), volumen_total), 6),
         "media_rate": f["media_rate"], "media_tot_size": f["media_tot_size"],
         "media_size_kb": f["media_size_kb"], "media_iat": f["media_iat"],
         "media_syn": f["media_syn"], "media_ack": f["media_ack"],
         "media_rst": f["media_rst"], "media_total_flags": f["media_total_flags"],
         "protocolos_distintos": int(f["protocolos_distintos"])}
        for posicion, (_, f) in enumerate(resumen.iterrows(), start=1)
    ]
    csv("10_resumen_consolidado.csv",
        ["posicion", "Label", "registros", "pct_registros", "volumen_bytes",
         "pct_volumen", "media_rate", "media_tot_size", "media_size_kb", "media_iat",
         "media_syn", "media_ack", "media_rst", "media_total_flags",
         "protocolos_distintos"],
        registros)
    return registros


consolidado = medir("10", "Resumen consolidado por clase", op10)
pd.DataFrame(consolidado).head(10)

  -> [10] Resumen consolidado por clase: 8.385 s


,posicion,Label,registros,pct_registros,volumen_bytes,pct_volumen,media_rate,media_tot_size,media_size_kb,media_iat,media_syn,media_ack,media_rst,media_total_flags,protocolos_distintos
0,1,DDoS-ICMP_Flood,92356,15.393000,5.610000e+06,7.108233,39957.267678,60.743213,0.059320,0.000108,0.000330,0.001629,0.000048,0.002606,4
1,2,DDoS-UDP_Flood,69419,11.570084,4.218632e+06,5.345279,33345.355260,60.770564,0.059346,0.000063,0.000589,0.002523,0.000074,0.004228,4
2,3,DDoS-TCP_Flood,57687,9.614708,3.631696e+06,4.601593,32271.151585,62.955185,0.061480,0.000061,0.000611,0.002596,0.000057,0.004300,2
3,4,DDoS-PSHACK_FLOOD,52521,8.753690,3.175767e+06,4.023902,32611.295328,60.466610,0.059049,0.000056,0.000368,0.967409,0.031412,1.964750,2
4,5,DDoS-SYN_Flood,52065,8.677688,3.247847e+06,4.115232,28803.489591,62.380613,0.060919,0.000071,0.986430,0.016452,0.008175,1.012184,2
5,6,DDoS-RSTFINFLOOD,51887,8.648021,3.176900e+06,4.025337,33311.555872,61.227279,0.059792,0.000366,0.000336,0.002020,0.995544,1.994087,2
6,7,DDoS-SynonymousIP_Flood,46151,7.692000,2.804484e+06,3.553462,32035.141131,60.767560,0.059343,0.000062,0.996234,0.001560,0.000068,0.998458,2
7,8,DoS-UDP_Flood,39414,6.569142,3.516836e+06,4.456058,20014.109743,89.228085,0.087137,0.060188,0.000858,0.005655,0.000136,0.008965,3
8,9,DoS-TCP_Flood,34265,5.710957,2.162479e+06,2.740000,25445.809057,63.110424,0.061631,0.000708,0.000799,0.009931,0.004834,0.017312,2
9,10,DoS-SYN_Flood,26023,4.337261,1.625435e+06,2.059531,22568.571802,62.461477,0.060998,0.000124,0.953776,0.042410,0.034673,1.033366,3


## Resumen de la ejecución

La tabla muestra el tiempo de cada operación. El total incluye la carga del CSV.

In [14]:
total = round(sum(f["segundos"] for f in TIEMPOS), 3)
pd.DataFrame(TIEMPOS)

csv("00_tiempos.csv", ["motor", "id_operacion", "operacion", "segundos"], TIEMPOS)
json_(
    "00_resumen_ejecucion.json",
    {"motor": MOTOR, "version": VERSION, "particiones": particiones,
     "bloque": BLOQUE, "hilos": HILOS, "dataset": CSV_MUESTRA.name,
     "filas_cargadas": len(limpio), "columnas": len(config.COLUMNAS),
     "operaciones": 10, "segundos_totales": total, "tiempos": TIEMPOS},
)
print(f"Tiempo total de las 10 operaciones: {total:.3f} s")
print(f"Archivos escritos en: {SALIDA}")

Tiempo total de las 10 operaciones: 60.496 s
Archivos escritos en: C:\Users\steve\Desktop\Big_Data\ProyectoBigData_CICIoT2023\results\fase4\dask
